In [1]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score, confusion_matrix
import mlflow
import mlflow.sklearn
import psutil
import time

In [2]:
# Set the MLflow tracking URI to 'http'
mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [3]:
from pathlib import Path
parent_path = file_path = Path.cwd().parent.parent
# print(Path.cwd())
# print(parent_path)

data = pd.read_csv(parent_path / 'data/preprocessed_data.csv')
data.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,MonthlyCharges_bin,TotalCharges_bin,tenure_bin
0,0,0,1,0,0.013889,0,1,0,0,2,...,0,0,1,2,0.115423,0.003437,0,1,1,1
1,1,0,0,0,0.472222,1,0,0,2,0,...,0,1,0,3,0.385075,0.217564,0,2,2,2
2,1,0,0,0,0.027778,1,0,0,2,2,...,0,0,1,3,0.354229,0.012453,1,2,1,1
3,1,0,0,0,0.625000,0,1,0,2,0,...,0,1,0,0,0.239303,0.211951,0,1,2,2
4,0,0,0,0,0.027778,1,0,1,0,0,...,0,0,1,2,0.521891,0.017462,1,2,1,1


In [4]:
data.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn', 'MonthlyCharges_bin',
       'TotalCharges_bin', 'tenure_bin'],
      dtype='object')

In [5]:
def preprocess_data(data):

    data = pd.read_csv(parent_path / 'data/preprocessed_data.csv')

    # Split data into X (features) and y (target)
    X = data.drop('Churn', axis=1)
    y = data['Churn']

    return X, y

In [6]:
X, y = preprocess_data(data)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [8]:
def train_model(X_train, y_train, max_depth=3, n_estimators=100):
    # Initialize the classifier
    clf = RandomForestClassifier(max_depth=max_depth, n_estimators=n_estimators, random_state=42)

    # Train the model
    clf.fit(X_train, y_train)

    return clf

In [9]:
clf = train_model(X_train, y_train, max_depth=3, n_estimators=100)

In [10]:
# Function to evaluate the model
def evaluate_model(model, X_test, y_test):
    # Make predictions
    y_pred = model.predict(X_test)

    # Evaluate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy:.2f}")

    # Display classification report
    print("Classification Report:")
    print(classification_report(y_test, y_pred))

# Function to log model and system metrics to MLflow
def log_to_mlflow(model, X_train, X_test, y_train, y_test):
    print(1)
    with mlflow.start_run():
        print(2)
        # Log hyper parameters used in Random Forest Algorithm
        mlflow.log_param("max_depth", model.max_depth)
        mlflow.log_param("n_estimators", model.n_estimators)
        print('log param')
        # Log model metrics
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='macro')
        recall = recall_score(y_test, y_pred, average='macro')
        f1 = f1_score(y_test, y_pred, average='macro')
        confusion = confusion_matrix(y_test, y_pred)
        
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1-score", f1)
        print('log_metric')       
        # Log confusion matrix
        confusion_dict = {
            "true_positive": confusion[1][1],
            "false_positive": confusion[0][1],
            "true_negative": confusion[0][0],
            "false_negative": confusion[1][0]
        }
        mlflow.log_metrics(confusion_dict)

        # Log system metrics
        # Example: CPU and Memory Usage
        cpu_usage = psutil.cpu_percent(interval=1)
        memory_usage = psutil.virtual_memory().percent

        mlflow.log_metric("system_cpu_usage", cpu_usage)
        mlflow.log_metric("system_memory_usage", memory_usage)

        # Log execution time for training the model
        execution_time = {}  # Dictionary to store execution times for different stages
        # Example: Execution time for training the model
        start_time = time.time()
        model = train_model(X_train, y_train)
        end_time = time.time()
        execution_time["system_model_training"] = end_time - start_time

        # Log execution time 
        mlflow.log_metrics(execution_time)
        print('log_metric last')
        # Evaluate model and log metrics
        evaluate_model(model, X_test, y_test)

        # Log model
        mlflow.sklearn.log_model(model, "model")


In [11]:
# Evaluate and log to MLflow
log_to_mlflow(clf, X_train, X_test, y_train, y_test)

1
2
log param
log_metric


2026/04/05 10:59:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


log_metric last
Accuracy: 0.79
Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.94      0.86      1539
           1       0.70      0.37      0.49       574

    accuracy                           0.79      2113
   macro avg       0.75      0.66      0.68      2113
weighted avg       0.77      0.79      0.76      2113



2026/04/05 10:59:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run zealous-worm-797 at: http://127.0.0.1:5000/#/experiments/0/runs/7d28202574b74ed79185e724f87c2044
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0
